In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Delete unneccessary columns for both male and female datasets
"Unnamed: 0", "Gender" for both male and female
"Pregnancies" for male

In [1]:
import pandas as pd

female_data = pd.read_csv('/kaggle/input/female-dataset/female_data.csv')
male_data = pd.read_csv('/kaggle/input/male-dataset/male_data.csv')


female_data.drop(['Unnamed: 0', 'Gender'], axis=1, inplace=True)
male_data.drop(['Unnamed: 0', 'Pregnancies', 'Gender'], axis=1, inplace=True)

Round numbers in columns 'PhysicallyActive", "Smoking", and "BPLevel"

In [2]:
female_data[['PhysicallyActive', 'Smoking', 'BPLevel', 'PhysHlth', 'MentHlth']] = female_data[['PhysicallyActive', 'Smoking', 'BPLevel', 'PhysHlth', 'MentHlth']].round()
male_data[['PhysicallyActive', 'Smoking', 'BPLevel', 'PhysHlth', 'MentHlth']] = male_data[['PhysicallyActive', 'Smoking', 'BPLevel', 'PhysHlth', 'MentHlth']].round()

Add a Patient_ID column for the female and male datasets

In [3]:
female_data['Patient_ID'] = range(1, len(female_data) + 1)
male_data['Patient_ID'] = range(len(female_data) + 1, len(female_data) + len(male_data) + 1)

Add a column 'Pregnant' for every value of '3' (Gestational Diabetes) in column 'Diabetes_Status'

In [4]:
import numpy as np
female_data['Pregnant'] = np.where(female_data['Diabetes_Status'] == 3, 1, 0)

Save Female and Male Datasets to csv files

In [5]:
female_data.to_csv('female_data.csv', index=False)
male_data.to_csv('male_data.csv', index=False)

Choosing features for female dataset based on SHAP values

In [8]:
features_not_chosen = ['Alopecia', 'Glucose', 'SkinThickness', 'Insulin', 'DiabetesPedigreeFunction', 'Diabetes_Status', 'Family_Diabetes', 'Smoking', 'Alcohol', 'RegularMedicine', 'JunkFood', 'Stress', 'Pdiabetes', 'UriationFreq', 'Diabetic', 'HighChol', 'CholCheck', 'Stroke', 'HeartDiseaseorAttack', 'Fruits', 'Veggies', 'AnyHealthcare', 'NoDocbcCost', 'MentHlth', 'unexplained prenetal loss', 'Large Child or Birth Default', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia', 'Genital thrush', 'Itching', 'Irritability', 'muscle stiffness', 'Patient_ID']
X_female = female_data.drop(columns=features_not_chosen)
y_female = female_data['Diabetes_Status']

from sklearn.model_selection import train_test_split

# Split the data (70% train, 30% test)
X_train_female, X_test_female, y_train_female, y_test_female = train_test_split(X_female, y_female, test_size=0.3, random_state=42)

# apply SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_female_smote, y_train_female_smote = smote.fit_resample(X_train_female, y_train_female)



Choosing features for male dataset based on SHAP values

In [9]:
features_not_chosen = ['Alopecia', 'Diabetes_Status', 'Family_Diabetes', 'Smoking', 'Alcohol', 'RegularMedicine', 'JunkFood', 'Stress', 'Pdiabetes', 'UriationFreq', 'Diabetic', 'HighChol', 'CholCheck', 'Stroke', 'HeartDiseaseorAttack', 'Fruits', 'Veggies', 'AnyHealthcare', 'NoDocbcCost', 'MentHlth', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia', 'Genital thrush', 'Itching', 'Irritability', 'muscle stiffness', 'Patient_ID']
X_male = male_data.drop(columns=features_not_chosen)
y_male = male_data['Diabetes_Status']



# Split the data (70% train, 30% test)
X_train_male, X_test_male, y_train_male, y_test_male = train_test_split(X_male, y_male, test_size=0.3, random_state=42)

X_train_male_smote, y_train_male_smote = smote.fit_resample(X_train_male, y_train_male)


In [10]:
print(X_train_female_smote.shape)
print(X_test_female.shape)

(300772, 22)
(38468, 22)


In [11]:
print(X_train_male_smote.shape)
print(X_test_male.shape)

(196119, 17)
(34156, 17)


Model Training

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler, LabelBinarizer

# Scale data for SGD and Gradient Boosting only (other models will be trained on the original data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_female_smote)
X_test_scaled = scaler.transform(X_test_female)

# Define models to evaluate
models = {
    'XGBoost': XGBClassifier(eval_metric='mlogloss', use_label_encoder=False),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Linear SVM': SVC(max_iter=2000, probability=True, kernel='linear'),  # Using LinearSVC for faster linear SVM
    'SGD Logistic Regression': SGDClassifier(loss='log', max_iter=2000, tol=1e-3, random_state=42),
}

# To store metrics for plotting
train_accuracy = {}
train_loss = {}

# Train and evaluate each model
for name, model in models.items():
    print(f'Training {name}...')
    
    # Choose scaled or unscaled data depending on the model
    if name == 'SGD Logistic Regression' or name == 'Gradient Boosting':
        X_train = X_train_scaled
        X_test = X_test_scaled
    else:
        X_train = X_train_female_smote
        X_test = X_test_female

    # Fit the model
    model.fit(X_train, y_train_female_smote)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # For log_loss, use probability predictions if available
    if hasattr(model, 'predict_proba'):
        y_train_prob = model.predict_proba(X_train)
        train_loss_val = log_loss(y_train_female_smote, y_train_prob)
    else:
        # If `predict_proba` is not available, set log loss to NaN
        train_loss_val = np.nan
    
    # Calculate accuracy
    train_acc = accuracy_score(y_train_female_smote, y_train_pred)
    
    train_accuracy[name] = train_acc
    train_loss[name] = train_loss_val
    
    print(f'Accuracy for {name} on training set: {train_acc:.2f}')
    if not np.isnan(train_loss_val):
        print(f'Log Loss for {name} on training set: {train_loss_val:.2f}')
    else:
        print(f'Log Loss for {name} on training set: Not Available (No predict_proba)')

# Plotting accuracy and loss
labels = list(train_accuracy.keys())
accuracy_values = list(train_accuracy.values())
loss_values = list(train_loss.values())

x = np.arange(len(labels))

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot accuracy
ax1.bar(x - 0.2, accuracy_values, 0.4, label='Train Accuracy', color='b')
ax1.set_ylabel('Accuracy', color='b')
ax1.set_ylim(0, 1)
ax1.set_xticks(x)
ax1.set_xticklabels(labels)

# Create a second y-axis to plot loss
ax2 = ax1.twinx()
ax2.bar(x + 0.2, loss_values, 0.4, label='Train Log Loss', color='r')
ax2.set_ylabel('Log Loss', color='r')

fig.tight_layout()
plt.title('Training Accuracy and Log Loss for Different Models (Multiclass)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.savefig('model_accuracies_female.png')
plt.show()



Training XGBoost...


KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler, LabelBinarizer

# Scale data for SGD and Gradient Boosting only (other models will be trained on the original data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_male_smote)
X_test_scaled = scaler.transform(X_test_male)

# Define models to evaluate
models = {
    'XGBoost': XGBClassifier(eval_metric='mlogloss', use_label_encoder=False),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Linear SVM': SVC(max_iter=2000, probability=True, kernel='linear'),  # Using LinearSVC for faster linear SVM
    'SGD Logistic Regression': SGDClassifier(loss='log', max_iter=2000, tol=1e-3, random_state=42),
}

# To store metrics for plotting
train_accuracy = {}
train_loss = {}

# Train and evaluate each model
for name, model in models.items():
    print(f'Training {name}...')
    
    # Choose scaled or unscaled data depending on the model
    if name == 'SGD Logistic Regression' or name == 'Gradient Boosting':
        X_train = X_train_scaled
        X_test = X_test_scaled
    else:
        X_train = X_train_male_smote
        X_test = X_test_male

    # Fit the model
    model.fit(X_train, y_train_male_smote)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # For log_loss, use probability predictions if available
    if hasattr(model, 'predict_proba'):
        y_train_prob = model.predict_proba(X_train)
        train_loss_val = log_loss(y_train_male_smote, y_train_prob)
    else:
        # If `predict_proba` is not available, set log loss to NaN
        train_loss_val = np.nan
    
    # Calculate accuracy
    train_acc = accuracy_score(y_train_male_smote, y_train_pred)
    
    train_accuracy[name] = train_acc
    train_loss[name] = train_loss_val
    
    print(f'Accuracy for {name} on training set: {train_acc:.2f}')
    if not np.isnan(train_loss_val):
        print(f'Log Loss for {name} on training set: {train_loss_val:.2f}')
    else:
        print(f'Log Loss for {name} on training set: Not Available (No predict_proba)')

# Plotting accuracy and loss
labels = list(train_accuracy.keys())
accuracy_values = list(train_accuracy.values())
loss_values = list(train_loss.values())

x = np.arange(len(labels))

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot accuracy
ax1.bar(x - 0.2, accuracy_values, 0.4, label='Train Accuracy', color='b')
ax1.set_ylabel('Accuracy', color='b')
ax1.set_ylim(0, 1)
ax1.set_xticks(x)
ax1.set_xticklabels(labels)

# Create a second y-axis to plot loss
ax2 = ax1.twinx()
ax2.bar(x + 0.2, loss_values, 0.4, label='Train Log Loss', color='r')
ax2.set_ylabel('Log Loss', color='r')

fig.tight_layout()
plt.title('Training Accuracy and Log Loss for Different Models (Multiclass)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.savefig('model_accuracies_male.png')
plt.show()
